# TP3 : Itération et évaluation d'un prompt

**Contexte :** assistant du service client de Lagune Banque. Les demandes sont classées en `regularisation_virement`, `remboursement`, `assistance` ou `information`.

Vous évaluerez trois versions du prompt sur 24 messages annotés. À chaque étape, comparez les prédictions aux réponses attendues, examinez les erreurs et modifiez le prompt sans perdre de vue son coût en tokens.

Complétez les cellules `TODO`, puis utilisez le contrôle automatique en fin de notebook.

**Sommaire**
- [Configuration](#configuration)
- [Données](#donnees)
- [Banc d'essai](#banc-essai)
- [Prompt minimal](#prompt-minimal)
- [Prompt avec définitions](#prompt-definitions)
- [Réduction du prompt](#reduction)
- [Comparaison](#comparaison)
- [Conclusion](#conclusion)

## <a id="configuration"></a>Configuration

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import litellm
import pandas as pd

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage"), Path("../../../code_partage"), Path("../../code_partage")] if (p / "config.py").exists()).resolve()))
from config import prepare

SEED = 42

In [ ]:
prepare()

## <a id="donnees"></a>Données

Le jeu de test contient 24 messages annotés. Huit exemples supplémentaires, distincts du jeu de test, peuvent être ajoutés au prompt. La catégorie correspond à l'objectif du client, et non à sa formule de politesse.

In [ ]:
dossier = next(p for p in [Path("01_fondamentaux_prompting/tp3_iterate_prompt/data"), Path("data"), Path("../../01_fondamentaux_prompting/tp3_iterate_prompt/data")] if (p / "messages.json").exists())
DONNEES = json.loads((dossier / "messages.json").read_text(encoding="utf-8"))
TEST, EXEMPLES = DONNEES["test"], DONNEES["exemples"]
CHOIX = ["regularisation_virement", "remboursement", "assistance", "information"]

pd.set_option("display.max_colwidth", 100)
pd.DataFrame(TEST).set_index("id")

## <a id="banc-essai"></a>Banc d'essai

`classify` applique un prompt aux 24 messages et renvoie les catégories prédites, les tokens d'entrée par message et le coût. Les exemples few-shot sont transmis sous forme de paires message et catégorie attendue.

**TODO 1** : écrivez `accuracy`, le nombre de demandes justes.

In [ ]:
def accuracy(predictions, expected):
    # TODO 1 : le nombre de prédictions égales à la catégorie attendue
    return 0


def few_shot(examples):
    return [{"role": role, "content": content} for message, demande in examples for role, content in (("user", message), ("assistant", demande))]


def classify(system, examples=()):
    predictions, tokens = [], 0
    for m in TEST:
        r = litellm.completion(model=os.environ["MODELE"], api_key=os.environ["LLM_API_KEY"], seed=SEED, num_retries=4, max_tokens=20,
                               messages=[{"role": "system", "content": system}, *few_shot(examples), {"role": "user", "content": m["message"]}])
        reponse = (r.choices[0].message.content or "").strip().lower()
        predictions.append(next((c for c in CHOIX if c in reponse), "?"))
        tokens += r.usage.prompt_tokens
    expected = [m["demande"] for m in TEST]
    return {"justes /24": accuracy(predictions, expected), "tokens d'entrée par message": round(tokens / len(TEST)), "predictions": predictions}


def errors(resultat):
    return pd.DataFrame([{"id": m["id"], "message": m["message"][:70], "attendu": m["demande"], "obtenu": p} for m, p in zip(TEST, resultat["predictions"]) if p != m["demande"]]).set_index("id") if resultat["justes /24"] < len(TEST) else "aucune erreur"

## <a id="prompt-minimal"></a>Prompt minimal

Cette version nomme les catégories sans les définir. Elle sert de référence pour les versions suivantes.

In [ ]:
PROMPT_V1 = f"Quelle est la demande de ce client ? Réponds uniquement par l'une de ces valeurs : {', '.join(CHOIX)}."
V1 = classify(PROMPT_V1)
{k: v for k, v in V1.items() if k != "predictions"}

In [ ]:
errors(V1)

## <a id="prompt-definitions"></a>Prompt avec rôle et définitions

Examinez les erreurs de la version 1. Ajoutez un rôle et une définition pour chaque catégorie afin de réduire les confusions.

**TODO 2** : écrivez `PROMPT_V2` : un rôle, une définition par valeur, et la consigne de répondre uniquement par la valeur.

In [ ]:
PROMPT_V2 = ""
# TODO 2 : rôle (conseiller du support client de Lagune Banque), une ligne de définition par valeur, réponse par une seule valeur
V2 = classify(PROMPT_V2)
{k: v for k, v in V2.items() if k != "predictions"}

In [ ]:
errors(V2)

## <a id="reduction"></a>Réduction du prompt

Le prompt est renvoyé à chaque appel. À partir de la version 2, gardez les éléments utiles à la classification et retirez le texte superflu. Vérifiez que la précision reste la même tout en réduisant le nombre de tokens.

**TODO 3** : écrivez `PROMPT_V3`, plus court que `PROMPT_V2`, sans réduire le nombre de réponses correctes sur les 24 messages. Les exemples few-shot sont possibles, mais augmentent le coût.

In [ ]:
PROMPT_V3 = PROMPT_V2
# TODO 3 : un prompt plus court que PROMPT_V2, qui garde les 24 bonnes réponses
V3 = classify(PROMPT_V3)
{k: v for k, v in V3.items() if k != "predictions"}

In [ ]:
errors(V3)

## <a id="comparaison"></a>Comparaison des versions

Le tableau compare le nombre de classifications correctes et le nombre moyen de tokens d'entrée par message pour chaque version.

In [ ]:
tableau = pd.DataFrame({"1. minimal": V1, "2. rôle et définitions": V2, "3. prompt raccourci": V3}).T.drop(columns="predictions")
tableau["coût / 1 000 messages (USD)"] = [round(sum(litellm.cost_per_token(model=os.environ["MODELE"], prompt_tokens=int(t * 1000), completion_tokens=5 * 1000)), 3) for t in tableau["tokens d'entrée par message"]]
tableau

## <a id="conclusion"></a>Conclusion

En deux ou trois phrases, indiquez la version retenue, le gain observé après chaque changement et le coût en tokens par message. Sur 24 exemples, un écart d'une seule réponse reste une indication, pas une preuve statistique.

**TODO 4**

In [ ]:
CONCLUSION = ""

## Contrôle automatique

La dernière cellule affiche ÉCHEC tant que les `# TODO` ne sont pas faits.

In [ ]:
try:
    assert accuracy(["a", "b", "c"], ["a", "x", "c"]) == 2, "accuracy : le nombre de prédictions justes (TODO 1)"
    assert PROMPT_V2.strip() and PROMPT_V2 != PROMPT_V1, "PROMPT_V2 : écrivez la version 2 (TODO 2)"
    assert V3["tokens d'entrée par message"] < V2["tokens d'entrée par message"], "PROMPT_V3 doit être plus court que PROMPT_V2 (TODO 3)"
    assert V3["justes /24"] >= V2["justes /24"], f"la version 3 ({V3['justes /24']}/24) ne doit pas faire moins bien que la version 2 ({V2['justes /24']}/24)"
    assert len(CONCLUSION) >= 60 and any(c.isdigit() for c in CONCLUSION), "écrivez une conclusion avec au moins un chiffre du tableau (TODO 4)"
    print("OK")
except Exception as e:
    print(f"ÉCHEC : {e if isinstance(e, AssertionError) else 'terminez les # TODO et relancez les cellules.'}")